# TravelMind RAG
## LangChain + OpenAI Embeddings + ChromaDB + Gradio

This notebook upgrades TravelMind from a prompt-only itinerary generator into a **Retrieval-Augmented Generation (RAG)** application.

The system will:

1. Load a small Kerala travel knowledge base.
2. Split it into overlapping chunks.
3. Convert each chunk into an embedding.
4. Store the embeddings in ChromaDB.
5. Retrieve relevant chunks using semantic search.
6. Give the retrieved information to an OpenAI chat model.
7. Generate a personalised itinerary with source references.

> The included knowledge base is demonstration data. Verify live prices, opening hours, weather, transport and safety information before real travel.

## Architecture

Travel knowledge → chunks → embeddings → ChromaDB

User preferences → semantic search → relevant chunks → LLM → itinerary

This follows the same core process as the uploaded restaurant RAG lab, using current LangChain package paths.

## 1. Install dependencies

Run this cell once. Restart the notebook kernel if an import still fails after installation.

In [11]:
%pip install -qU langchain-openai langchain-chroma langchain-community langchain-text-splitters chromadb gradio python-dotenv tiktoken OpenAI

Note: you may need to restart the kernel to use updated packages.


## 2. Import the libraries

The modern integration packages are langchain_chroma, langchain_openai, langchain_community and langchain_text_splitters.

In [12]:
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage

print("Libraries imported successfully.")

Libraries imported successfully.


## 3. Load the OpenAI API key safely

The key is hidden while you type and is never printed. Do not hard-code an API key in a notebook uploaded to GitHub.

In [13]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key=openai_api_key)
print("OpenAI client successfully configured.")



OpenAI client successfully configured.


## 4. Create a demonstration travel knowledge base

The restaurant notebook reads a separate text file. To keep this project inside one notebook, this cell creates the text file automatically. Later, replace this sample with verified travel guides, PDFs or database records.

In [14]:
TRAVEL_DATA = """
# KOCHI
Kochi is a historic coastal city in Kerala combining colonial heritage, harbour scenery, arts and local cuisine. Fort Kochi is known for walkable heritage streets, colonial-era buildings, cafés and the Chinese fishing nets waterfront. Mattancherry includes the Mattancherry Palace area, spice-market streets and the Paradesi Synagogue area. Travellers should verify current entry times and weekly closure days before visiting heritage sites.

Kochi experiences include a Fort Kochi heritage walk, watching sunset near the waterfront, exploring Mattancherry, visiting museums or art spaces and trying Kerala seafood or vegetarian meals. Local movement options include metro services in parts of the urban area, buses, ferries, auto-rickshaws and app-based taxis. Travel time should include traffic buffers.

A relaxed two-day Kochi plan can dedicate one day to Fort Kochi and Mattancherry and another to the modern city, waterfront or nearby cultural experiences. Photography interests are best served by early-morning heritage streets and evening waterfront light.

# MUNNAR
Munnar is a hill destination known for tea-growing landscapes, viewpoints, cool-weather scenery and nature experiences. Road travel through the hills can be slow, so an itinerary should avoid packing too many distant attractions into one day. Weather, fog and rain may change visibility and travel time.

Common Munnar experiences include tea-garden viewpoints, a tea museum when open, nature walks and scenic drives. Eravikulam National Park is a popular nearby attraction, but visitors must check current opening status, seasonal closures, ticket rules and transport arrangements. Early starts can help with crowds and road conditions.

A practical Munnar visit normally needs at least two nights when travelling from Kochi. Budget travellers can use local restaurants and shared transport where available, but taxis are often more convenient for dispersed viewpoints.

# ALAPPUZHA
Alappuzha is associated with Kerala backwaters, canals, villages, paddy landscapes and houseboat tourism. Experiences include shikara rides, public ferries, canoe trips, village walks and houseboat stays. A public ferry or short shikara ride may suit travellers who want a lower-cost backwater experience.

Houseboat packages vary in route, duration, meals, boat quality and overnight facilities. Travellers should confirm inclusions, safety arrangements, boarding point and cancellation terms before booking. Day cruises can be easier to combine with a short itinerary than overnight stays.

Alappuzha can be reached by train, bus or road from Kochi and other Kerala cities. Actual journey time depends on traffic and service schedules. Visitors interested in photography may prefer early morning or late afternoon light along canals and paddy fields.

# VARKALA
Varkala is a coastal destination known for its cliff area, sea views, beach atmosphere, cafés and sunsets. Swimming conditions can vary, and travellers should follow lifeguard instructions, warning flags and local safety guidance. Monsoon conditions may affect beach access and cliff paths.

A short Varkala stay may include a cliff walk, beach time when safe, sunset viewing and local dining. Railway connections make Varkala accessible from several Kerala cities, while local movement can use auto-rickshaws and taxis.

# WAYANAD
Wayanad is a hill district with forest landscapes, plantations, viewpoints, waterfalls and trekking-related attractions. Destinations are spread across the district, so choosing accommodation near the planned attractions reduces road travel. Rain and local restrictions can affect trekking and waterfall access.

Travellers should check official advisories, opening conditions and guide requirements for nature attractions. A realistic plan groups nearby locations rather than attempting to cross the district repeatedly.

# KERALA TRAVEL PLANNING GUIDANCE
Kerala itineraries should account for monsoon rain, heat, road traffic, hill-road conditions and seasonal closures. Live weather, transport schedules, attraction timings and ticket prices must be checked using current official sources. Avoid presenting demonstration costs as guaranteed prices.

A balanced daily itinerary usually includes one major morning activity, lunch and rest time, one afternoon activity and flexible evening time. Leave additional travel buffers for hill routes and city traffic. Travellers with children, older adults or mobility constraints may need shorter walking periods and fewer daily activities.

Budget planning should separate accommodation, intercity transport, local transport, food, activities and an emergency buffer. The application should explain when the available budget may be unrealistic instead of inventing exact prices.

Food suggestions may include appam, puttu, idiyappam, Kerala meals, seafood where suitable and regional snacks. The planner should ask about allergies and dietary preferences and should not assume every restaurant can meet them.
"""

DATA_FILE_PATH = Path("travelmind_kerala_knowledge.txt")
DATA_FILE_PATH.write_text(TRAVEL_DATA.strip(), encoding="utf-8")

print(f"Knowledge base created: {DATA_FILE_PATH}")
print(f"Characters written: {len(TRAVEL_DATA):,}")

Knowledge base created: travelmind_kerala_knowledge.txt
Characters written: 5,019


## 5. Load the knowledge base

TextLoader converts the text file into a LangChain Document. A document contains page content plus metadata such as its source filename.

In [15]:
loader = TextLoader(str(DATA_FILE_PATH), encoding="utf-8")
raw_documents = loader.load()

print(f"Loaded {len(raw_documents)} document(s).")
print("Source metadata:", raw_documents[0].metadata)
print("\nPreview:\n")
print(raw_documents[0].page_content[:500])

Loaded 1 document(s).
Source metadata: {'source': 'travelmind_kerala_knowledge.txt'}

Preview:

# KOCHI
Kochi is a historic coastal city in Kerala combining colonial heritage, harbour scenery, arts and local cuisine. Fort Kochi is known for walkable heritage streets, colonial-era buildings, cafés and the Chinese fishing nets waterfront. Mattancherry includes the Mattancherry Palace area, spice-market streets and the Paradesi Synagogue area. Travellers should verify current entry times and weekly closure days before visiting heritage sites.

Kochi experiences include a Fort Kochi heritage w


## 6. Split the document into chunks

Embedding an entire guide as one vector would make retrieval imprecise. Chunking creates smaller units. The overlap repeats some text across boundaries so an important idea is less likely to be cut in half.

In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,
    separators=["\n# ", "\n\n", "\n", ". ", " "],
)

chunks = text_splitter.split_documents(raw_documents)

if not chunks:
    raise ValueError("No chunks were created.")

for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = index

print(f"Created {len(chunks)} chunks.")
print("\nExample chunk:\n")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

Created 10 chunks.

Example chunk:

# KOCHI
Kochi is a historic coastal city in Kerala combining colonial heritage, harbour scenery, arts and local cuisine. Fort Kochi is known for walkable heritage streets, colonial-era buildings, cafés and the Chinese fishing nets waterfront. Mattancherry includes the Mattancherry Palace area, spice-market streets and the Paradesi Synagogue area. Travellers should verify current entry times and weekly closure days before visiting heritage sites.

Metadata: {'source': 'travelmind_kerala_knowledge.txt', 'chunk_id': 0}


## 7. Create embeddings

An embedding is a numerical representation of meaning. Similar travel passages should produce vectors that are close together. This notebook uses text-embedding-3-small.

In [17]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

sample_embedding = embeddings.embed_query(
    "I want a peaceful Kerala backwater experience"
)

print("Embedding created successfully.")
print("Vector dimensions:", len(sample_embedding))
print("First 8 values:", sample_embedding[:8])

Embedding created successfully.
Vector dimensions: 1536
First 8 values: [0.00841522216796875, -0.001903533935546875, -0.040283203125, 0.0198822021484375, -0.038848876953125, -0.006023406982421875, 0.00843048095703125, 0.034393310546875]


## 8. Store the embeddings in ChromaDB

The persistence directory saves the vector database locally. The notebook checks whether the collection is empty before adding chunks, preventing duplicate insertion when this cell is rerun.

In [18]:
CHROMA_DIRECTORY = "./travelmind_chroma_db"
COLLECTION_NAME = "travelmind_kerala_guides"

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIRECTORY,
)

existing_count = vector_store._collection.count()

if existing_count == 0:
    chunk_ids = [f"travel-chunk-{i}" for i in range(len(chunks))]
    vector_store.add_documents(documents=chunks, ids=chunk_ids)
    print(f"Added {len(chunks)} chunks to ChromaDB.")
else:
    print(f"ChromaDB already contains {existing_count} chunks; insertion skipped.")

print("Stored vectors:", vector_store._collection.count())
print("Database directory:", CHROMA_DIRECTORY)

ChromaDB already contains 10 chunks; insertion skipped.
Stored vectors: 10
Database directory: ./travelmind_chroma_db


## 9. Test semantic search

Semantic search compares meaning, not only exact keywords. This query should retrieve Alappuzha passages even though it says calm waterways instead of backwaters.

In [19]:
test_query = "Where can I experience calm waterways on a small budget?"

search_results = vector_store.similarity_search_with_relevance_scores(
    test_query,
    k=3,
)

print("Query:", test_query)

for number, (document, score) in enumerate(search_results, start=1):
    print("\n" + "-" * 70)
    print(f"Result {number} | Relevance: {score:.3f}")
    print(document.page_content[:600])
    print("Source:", document.metadata.get("source"))
    print("Chunk:", document.metadata.get("chunk_id"))

Query: Where can I experience calm waterways on a small budget?

----------------------------------------------------------------------
Result 1 | Relevance: 0.190
# ALAPPUZHA
Alappuzha is associated with Kerala backwaters, canals, villages, paddy landscapes and houseboat tourism. Experiences include shikara rides, public ferries, canoe trips, village walks and houseboat stays. A public ferry or short shikara ride may suit travellers who want a lower-cost backwater experience.

Houseboat packages vary in route, duration, meals, boat quality and overnight facilities. Travellers should confirm inclusions, safety arrangements, boarding point and cancellation terms before booking. Day cruises can be easier to combine with a short itinerary than overnight sta
Source: travelmind_kerala_knowledge.txt
Chunk: 4

----------------------------------------------------------------------
Result 2 | Relevance: 0.059
# WAYANAD
Wayanad is a hill district with forest landscapes, plantations, viewpoints, 

## 10. Create the RAG itinerary function

The function performs two stages:

1. Retrieval: ChromaDB finds passages relevant to the traveller.
2. Generation: The chat model receives those passages as context and writes the itinerary.

The prompt instructs the model not to invent facts outside the retrieved knowledge.

In [20]:
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0.3,
)

def format_documents(documents):
    """Combine retrieved chunks into clearly numbered context blocks."""
    blocks = []
    for number, document in enumerate(documents, start=1):
        source = document.metadata.get("source", "Unknown source")
        chunk_id = document.metadata.get("chunk_id", "Unknown")
        blocks.append(
            f"[Source {number}: {source}, chunk {chunk_id}]\n"
            f"{document.page_content}"
        )
    return "\n\n".join(blocks)

def create_rag_itinerary(destination, days, budget, traveller_type, interests):
    """Retrieve destination knowledge and generate a grounded itinerary."""
    destination = str(destination).strip()
    budget = str(budget).strip()
    traveller_type = str(traveller_type).strip()

    if isinstance(interests, list):
        interests_text = ", ".join(interests)
    else:
        interests_text = str(interests).strip()

    if not destination:
        return "Please enter a destination.", ""

    retrieval_query = (
        f"Destination: {destination}. "
        f"Trip duration: {days} days. "
        f"Budget: {budget}. "
        f"Traveller: {traveller_type}. "
        f"Interests: {interests_text}. "
        "Find relevant attractions, transport guidance, pacing, food and safety information."
    )

    retrieved_documents = vector_store.similarity_search(retrieval_query, k=4)
    context = format_documents(retrieved_documents)

    system_prompt = """
You are TravelMind, a careful Kerala itinerary assistant using Retrieval-Augmented Generation.

Rules:
1. Base destination facts on the supplied retrieved context.
2. Do not invent exact prices, opening hours, journey times or current availability.
3. Tell the traveller to verify live schedules, weather, prices and closures.
4. If the destination is not adequately covered by the context, say so.
5. Respect the number of days, budget, traveller type and interests.
6. Produce a practical plan rather than filling every hour.

Output sections:
- Trip overview
- Day-by-day itinerary
- Budget guidance
- Food suggestions
- Transport guidance
- Important checks before travel
"""

    user_prompt = f"""
Create a {int(days)}-day itinerary.

TRAVELLER REQUEST
Destination: {destination}
Budget: {budget}
Traveller type: {traveller_type}
Interests: {interests_text}

RETRIEVED KNOWLEDGE
{context}
"""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ])

    source_lines = []
    for document in retrieved_documents:
        source_lines.append(
            f"{document.metadata.get('source', 'Unknown source')} "
            f"(chunk {document.metadata.get('chunk_id', 'Unknown')})"
        )

    unique_sources = list(dict.fromkeys(source_lines))
    sources_text = "\n".join(f"• {source}" for source in unique_sources)

    return response.content, sources_text

print("RAG itinerary function created.")

RAG itinerary function created.


## 11. Test the full RAG pipeline

In [21]:
test_itinerary, test_sources = create_rag_itinerary(
    destination="Alappuzha",
    days=2,
    budget="₹15,000 for two people",
    traveller_type="Couple",
    interests=["Backwaters", "Photography", "Kerala food"],
)

print("TRAVELMIND ITINERARY\n")
print(test_itinerary)
print("\nRETRIEVED SOURCES\n")
print(test_sources)

TRAVELMIND ITINERARY

### Trip Overview
Welcome to Alappuzha, the heart of Kerala's backwaters! This 2-day itinerary is designed for a couple interested in exploring the serene backwaters, capturing beautiful photographs, and indulging in authentic Kerala cuisine. With a budget of ₹15,000 for two people, you'll experience the best of Alappuzha without overspending.

### Day-by-Day Itinerary

#### Day 1: Backwater Exploration
- **Morning**: Start your day with a shikara ride through the backwaters. This is a more affordable option than a houseboat stay and offers stunning views of the canals and paddy fields. Aim for an early morning ride for the best light for photography.
  
- **Lunch**: Enjoy a traditional Kerala meal at a local restaurant. Look for places that serve authentic dishes like Kerala Sadya (a feast served on a banana leaf) or fresh seafood.

- **Afternoon**: After lunch, take a leisurely stroll through the nearby villages. Engage with locals and capture the essence of rur

## 12. Build the Gradio interface

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="TravelMind RAG") as demo:
    gr.Markdown(
        """
        # 🌴 TravelMind RAG
        ### Kerala itinerary planning with LangChain, OpenAI embeddings and ChromaDB
        The assistant retrieves relevant information from its travel knowledge base before creating your plan.
        """
    )

    with gr.Row():
        destination_input = gr.Dropdown(
            choices=["Kochi", "Munnar", "Alappuzha", "Varkala", "Wayanad"],
            value="Kochi",
            label="Destination",
        )
        days_input = gr.Slider(
            minimum=1,
            maximum=7,
            value=3,
            step=1,
            label="Number of days",
        )

    with gr.Row():
        budget_input = gr.Textbox(
            value="₹20,000 for two people",
            label="Budget",
        )
        traveller_input = gr.Dropdown(
            choices=["Solo", "Couple", "Family", "Friends"],
            value="Couple",
            label="Traveller type",
        )

    interests_input = gr.CheckboxGroup(
        choices=[
            "History",
            "Nature",
            "Backwaters",
            "Beaches",
            "Photography",
            "Kerala food",
            "Relaxation",
        ],
        value=["Nature", "Kerala food"],
        label="Interests",
    )

    create_button = gr.Button("Create RAG Itinerary", variant="primary")
    itinerary_output = gr.Markdown(label="Itinerary")
    sources_output = gr.Textbox(
        label="Retrieved sources",
        lines=4,
        interactive=False,
    )

    create_button.click(
        fn=create_rag_itinerary,
        inputs=[
            destination_input,
            days_input,
            budget_input,
            traveller_input,
            interests_input,
        ],
        outputs=[itinerary_output, sources_output],
    )


demo.launch(
    share=True,
    debug=True
)

C:\Users\HP\AppData\Local\Temp\ipykernel_4652\3093091394.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="TravelMind RAG") as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d6b64dc183dd21104f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
